
### WEEK 3: THE CLEANING SPRINT

#### Project Title
Spotify Music Trends and Audio Features Analysis Using Data Science and Visualization

#### Course
24ADI204 – Data Science and Visualization

#### Team
DSV Team 9

#### Aim
To identify and handle missing values, remove duplicate records, detect potential outliers, and prepare the Spotify dataset for Exploratory Data Analysis.

#### Objectives
1. Load the original Spotify dataset.
2. Examine data types and descriptive statistics.
3. Identify missing values in the dataset.
4. Apply median and mode imputation.
5. Detect and remove exact duplicate records.
6. Visualize potential outliers using boxplots.
7. Detect outliers using IQR and Z-score methods.
8. Compare the results of both outlier detection methods.
9. Export the cleaned dataset for Week 4.

#### Tools Used
- Python
- Pandas
- NumPy
- Matplotlib
- Seaborn
- Jupyter Notebook



### 1. Importing Required Libraries

The following Python libraries are imported for data cleaning and analysis.

- **Pandas:** Used to load, manipulate, and clean the dataset.
- **NumPy:** Used for numerical calculations.
- **Matplotlib:** Used to create data visualizations.
- **Seaborn:** Used to generate statistical plots.
- **Pathlib:** Used to locate and manage dataset files.

These libraries provide the necessary tools for identifying and handling data-quality issues.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 100)


### 2. Loading the Original Spotify Dataset

The original Spotify dataset is loaded from the CSV file named `DSV_Team-9_Raw_Dataset.csv`.

The dataset contains 19,335 records and 46 columns.

The original dataset is stored separately as `df_raw`, while a working copy named `df` is created for cleaning.

This ensures that the original data remains unchanged throughout the preprocessing process.


In [ ]:
# Load the original Spotify dataset from your computer
from pathlib import Path

csv_path = Path("DSV_Team-9_Raw_Dataset.csv")

if not csv_path.is_file():
    print("Current notebook folder:", Path.cwd())
    csv_path = Path(input("Enter the full path of DSV_Team-9_Raw_Dataset.csv: ").strip().strip('"'))

if not csv_path.is_file():
    raise FileNotFoundError(f"CSV file not found: {csv_path}")

# Read the original 46-column dataset
df_raw = pd.read_csv(csv_path, encoding="latin1", low_memory=False)

# Create a working copy for cleaning
df = df_raw.copy()

# Create an output folder for the cleaned dataset
processed_dir = Path.cwd() / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

print("\nDataset loaded successfully!")
print("Original dataset shape:", df_raw.shape)
print("Working dataset shape:", df.shape)

display(df.head())



### 3. Understanding Dataset Structure

The dataset structure is examined before performing data cleaning.

The following functions are used:

- `df.shape` – Displays the number of rows and columns.
- `df.columns` – Displays the available column names.
- `df.info()` – Displays data types, non-null counts, and memory usage.
- `df.describe()` – Generates descriptive statistics for numerical attributes.

This analysis helps identify potential data-quality issues.


In [ ]:
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
df.info()


#### Preliminary Descriptive Statistics

Descriptive statistics provide a summary of the numerical attributes in the Spotify dataset.

The `describe()` function calculates the count, mean, standard deviation, minimum, maximum, and quartiles.

These statistics help identify unusual values and understand the initial distribution of numerical features before cleaning.


In [ ]:
df.describe(include='number').T


### 4. Identifying Missing Values

Missing values occur when information is unavailable for certain records.

The `isna()` function is used to identify missing values in each column.

The total number and percentage of missing values are calculated to understand the completeness of the dataset.

This analysis helps determine which columns require imputation or further investigation.


In [ ]:
missing_before = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(2)
}).sort_values("missing_count", ascending=False)

missing_before[missing_before["missing_count"] > 0]


### 5. Handling Missing Values Using Imputation

Imputation is a data-cleaning technique used to replace missing values with suitable estimates.

The following methods are applied:

- **Median Imputation:** Missing values in numerical columns are replaced with the median.
- **Mode Imputation:** Missing values in categorical columns are replaced with the most frequently occurring value.
- **Entirely Missing Columns:** Columns containing only missing values are removed.

Median imputation is less sensitive to extreme values than mean imputation.

Mode imputation preserves the most frequently occurring category.

These techniques help improve dataset completeness.


In [ ]:
all_missing_columns = df.columns[df.isna().all()].tolist()
df = df.drop(columns=all_missing_columns)

numeric_columns = df.select_dtypes(include=np.number).columns.tolist()
categorical_columns = df.select_dtypes(include=["object", "category", "string", "bool"]).columns.tolist()

numerical_fill_values = {}
for col in numeric_columns:
    if df[col].isna().any():
        value = df[col].median()
        if pd.notna(value):
            numerical_fill_values[col] = value
            df[col] = df[col].fillna(value)

categorical_fill_values = {}
for col in categorical_columns:
    if df[col].isna().any():
        modes = df[col].mode(dropna=True)
        if not modes.empty:
            value = modes.iloc[0]
            categorical_fill_values[col] = value
            df[col] = df[col].fillna(value)

print("Entirely empty columns removed:", all_missing_columns)
print("Numerical columns imputed:", list(numerical_fill_values))
print("Categorical columns imputed:", list(categorical_fill_values))


### 6. Verifying Missing Values After Imputation

After applying imputation techniques, the dataset is checked again for missing values.

The number and percentage of remaining missing values are calculated.

This verification helps determine whether the missing-value handling process was successful.


In [ ]:
missing_after = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(2)
}).sort_values("missing_count", ascending=False)

remaining_missing = missing_after[missing_after["missing_count"] > 0]
print("Columns still containing missing values:", len(remaining_missing))
remaining_missing


### 7. Identifying and Removing Duplicate Records

Duplicate records occur when identical rows appear more than once in a dataset.

The `duplicated()` function is used to identify exact duplicate rows.

The `drop_duplicates()` function removes repeated records from the working dataset.

Removing duplicates helps prevent repeated observations from affecting statistical analysis and visualizations.


In [ ]:
duplicates_before = int(df.duplicated().sum())
df = df.drop_duplicates().copy()

print("Duplicate rows found:", duplicates_before)
print("Shape after duplicate removal:", df.shape)
print("Duplicate rows remaining:", int(df.duplicated().sum()))


### 8. Selecting Numerical Spotify Audio Features

Numerical attributes are selected for outlier detection and statistical analysis.

The selected features include:

- **Popularity:** Represents the popularity score of a track.
- **Danceability:** Measures how suitable a song is for dancing.
- **Energy:** Represents the intensity of a song.
- **Loudness:** Measures the overall loudness in decibels.
- **Speechiness:** Measures the presence of spoken words.
- **Acousticness:** Estimates whether a song is acoustic.
- **Instrumentalness:** Estimates the likelihood of instrumental content.
- **Liveness:** Estimates the presence of a live audience.
- **Valence:** Represents the musical positivity of a song.
- **Tempo:** Represents the speed of a song in beats per minute.

Only available numerical columns are selected for further analysis.


In [ ]:
candidate_features = [
    "popularity", "danceability", "energy", "loudness",
    "speechiness", "acousticness", "instrumentalness",
    "liveness", "valence", "tempo"
]
column_lookup = {str(c).strip().lower(): c for c in df.columns}
audio_features = [
    column_lookup[name] for name in candidate_features
    if name in column_lookup and pd.api.types.is_numeric_dtype(df[column_lookup[name]])
]
if not audio_features:
    raise ValueError("No expected numerical Spotify features were found. Check the CSV column names.")

print("Features selected:", audio_features)
df[audio_features].describe().T


### 9. Outlier Detection Using Boxplots

A boxplot is a statistical visualization used to understand the distribution of numerical data.

It displays:

- **Median:** The middle value of the distribution.
- **First Quartile (Q1):** The 25th percentile.
- **Third Quartile (Q3):** The 75th percentile.
- **Interquartile Range (IQR):** The difference between Q3 and Q1.
- **Potential Outliers:** Values located beyond the boxplot whiskers.

Separate boxplots are generated for the selected Spotify numerical features.

This visualization helps identify unusual values that may require further investigation.


In [ ]:
for col in audio_features:
    plt.figure(figsize=(9, 3))
    sns.boxplot(x=df[col], color="skyblue")
    plt.title(f"Boxplot of {col}")
    plt.xlabel(col)
    plt.tight_layout()
    plt.show()


### 10. Outlier Detection Using the IQR Method

The Interquartile Range (IQR) method is used to identify potential outliers based on the middle 50% of the data.

The IQR is calculated using:

**IQR = Q3 - Q1**

The lower and upper boundaries are calculated as:

**Lower Bound = Q1 - 1.5 × IQR**

**Upper Bound = Q3 + 1.5 × IQR**

Values below the lower bound or above the upper bound are classified as potential outliers.

The method is applied to the selected numerical Spotify features.

The number and percentage of potential outliers are calculated for each feature.


In [ ]:
iqr_rows = []
for col in audio_features:
    values = df[col].dropna()
    q1, q3 = values.quantile(0.25), values.quantile(0.75)
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    mask = (df[col] < lower) | (df[col] > upper)
    iqr_rows.append({
        "feature": col, "Q1": q1, "Q3": q3, "IQR": iqr,
        "lower_bound": lower, "upper_bound": upper,
        "potential_outliers": int(mask.sum()),
        "outlier_percent": round(mask.mean() * 100, 2)
    })

iqr_summary = pd.DataFrame(iqr_rows).set_index("feature")
iqr_summary


### 11. Outlier Detection Using the Z-Score Method

The Z-score method measures how far a numerical value is from the mean in terms of standard deviations.

The Z-score is calculated using:

**Z = (X - Mean) / Standard Deviation**

Where:

- **X:** Individual observation.
- **Mean:** Average value of the feature.
- **Standard Deviation:** Measure of data variation.

In this analysis, values with an absolute Z-score greater than 3 are flagged as potential outliers.

The method is applied to the selected Spotify numerical features to identify extreme observations.


In [ ]:
z_rows = []
for col in audio_features:
    values = df[col]
    mean_value = values.mean()
    std_value = values.std(ddof=0)
    if pd.isna(std_value) or np.isclose(std_value, 0):
        z_scores = pd.Series(0.0, index=values.index)
    else:
        z_scores = (values - mean_value) / std_value
    mask = z_scores.abs() > 3
    z_rows.append({
        "feature": col, "mean": mean_value,
        "standard_deviation": std_value,
        "potential_outliers_|z|>3": int(mask.sum()),
        "outlier_percent": round(mask.mean() * 100, 2)
    })

zscore_summary = pd.DataFrame(z_rows).set_index("feature")
zscore_summary


### 12. Comparing IQR and Z-Score Results

The results of the IQR and Z-score methods are compared to understand differences in outlier detection.

The comparison report includes:

1. Feature name
2. Number of potential outliers detected using IQR
3. Percentage of potential outliers detected using IQR
4. Number of potential outliers detected using Z-score

The IQR method uses quartiles, while the Z-score method uses the mean and standard deviation.

This comparison helps determine which features require closer examination.


In [ ]:
outlier_comparison = iqr_summary[["potential_outliers", "outlier_percent"]].join(
    zscore_summary[["potential_outliers_|z|>3"]]
).rename(columns={
    "potential_outliers": "IQR_outlier_count",
    "outlier_percent": "IQR_outlier_percent",
    "potential_outliers_|z|>3": "Z_score_outlier_count"
})
outlier_comparison


### 13. Outlier Handling Decision

Outlier detection is an important step in data preprocessing.

However, not all outliers represent errors.

In the Spotify dataset, unusual values of tempo, loudness, acousticness, or instrumentalness may represent genuine musical characteristics.

Therefore, the following approach is adopted:

1. Identify potential outliers using boxplots.
2. Detect outliers using the IQR method.
3. Detect outliers using the Z-score method.
4. Compare the results of both methods.
5. Retain potential outliers unless there is evidence that the values are invalid.

This approach helps preserve meaningful information in the dataset.


In [ ]:
print("Rows retained after missing-value handling and duplicate removal:", len(df))
print("Columns retained:", df.shape[1])
print("No rows were removed solely because of IQR or Z-score flags.")


### 14. Final Data Quality Check

After completing the cleaning process, the dataset is examined again to verify its quality.

The final verification includes:

- Number of rows and columns
- Remaining missing values
- Remaining exact duplicate records
- Preview of cleaned records
- Descriptive statistics of selected numerical features

This step helps confirm that the dataset is ready for further analysis.


In [ ]:
print("Final cleaned shape:", df.shape)
print("Remaining missing cells:", int(df.isna().sum().sum()))
print("Remaining exact duplicate rows:", int(df.duplicated().sum()))
display(df.head())
display(df[audio_features].describe().T)


### 15. Exporting the Cleaned Spotify Dataset

After completing the cleaning process, the processed dataset is exported into CSV format.

The `to_csv()` function is used to save the cleaned dataset.

The output file is named `spotify_cleaned.csv` and is stored in the `data/processed` folder.

The exported file is loaded again to verify its dimensions.

This cleaned dataset can be used for Week 4 Exploratory Data Analysis.


In [ ]:
cleaned_path = processed_dir / "spotify_cleaned.csv"
df.to_csv(cleaned_path, index=False)

exported = pd.read_csv(cleaned_path, low_memory=False)
print("Saved to:", cleaned_path)
print("Exported shape:", exported.shape)


### 16. Conclusion

Data cleaning and preprocessing were performed on the Spotify Songs dataset using Python, Pandas, NumPy, Matplotlib, and Seaborn.

The dataset was examined to identify missing values, duplicate records, and potential outliers.

Missing values were handled using median imputation for numerical attributes and mode imputation for categorical attributes.

Exact duplicate records were checked and removed where present.

Potential outliers were identified using boxplots, the Interquartile Range method, and the Z-score method.

The results of both statistical methods were compared to understand differences in outlier detection.

Potential outliers were retained because extreme musical characteristics may represent valid Spotify tracks.

Finally, the processed dataset was verified and exported as a CSV file for further analysis.

#### Result

The Spotify dataset was cleaned, verified, and prepared for Exploratory Data Analysis. Missing-value handling, duplicate checking, and outlier detection were performed successfully.
